# Demo 02 : A first look at LangGraph

Run the cells in order and watch. There is nothing to write. The last cell of this notebook holds a game built by
the crew, and we can play it.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


We load the game kit. A game is one HTML file with a canvas. The kit holds the drawing, the keyboard and a bot that
plays. The `logic` block holds the rules, in plain JavaScript. The harness runs that block through the QuickJS
interpreter, so the tests and the bot play the real game without a browser.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

import importlib.util
from pathlib import Path

kit = Path(data_folder("game-studio/kit"))
spec = importlib.util.spec_from_file_location("harness", kit / "harness.py")
harness = importlib.util.module_from_spec(spec)
spec.loader.exec_module(harness)

TEMPLATE = (kit / "template.html").read_text(encoding="utf-8")
RULES = (kit / "rules.md").read_text(encoding="utf-8")
TWISTS = json.loads((kit / "twists.json").read_text(encoding="utf-8"))
BRIEFS = json.loads((kit / "briefs.json").read_text(encoding="utf-8"))

print("twists:", list(TWISTS))
print("briefs:", [b["id"] for b in BRIEFS])
print("logic block of the kit:", len(harness.logic_of(TEMPLATE).splitlines()), "lines")

twists: ['golden', 'wormhole', 'shield', 'clock']
briefs: ['second-chance', 'big-prize', 'shortcut', 'breather']
logic block of the kit: 76 lines


## a · first-graph

We build a graph of plain functions and no model. It takes a logic block, puts it in the kit, runs the tests, and
ships the build or sends it back to the Coder. We feed it three real builds.

In [3]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class Build(TypedDict, total=False):
    logic: str
    twist: str
    html: str
    passed: bool
    summary: str
    verdict: str


def assemble(state: Build) -> dict:
    return {"html": harness.splice(TEMPLATE, state["logic"])}


def test(state: Build) -> dict:
    report = harness.run_tests(state["html"], state.get("twist"))
    return {"passed": report["passed"], "summary": harness.render(report).splitlines()[0]}


def route(state: Build) -> Literal["ship", "back_to_coder"]:
    return "ship" if state["passed"] else "back_to_coder"


builder = StateGraph(Build)
builder.add_node("assemble", assemble)
builder.add_node("test", test)
builder.add_node("ship", lambda state: {"verdict": "ship"})
builder.add_node("back_to_coder", lambda state: {"verdict": "back to the Coder"})
builder.add_edge(START, "assemble")
builder.add_edge("assemble", "test")
builder.add_conditional_edges("test", route)
builder.add_edge("ship", END)
builder.add_edge("back_to_coder", END)
graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	assemble(assemble)
	test(test)
	ship(ship)
	back_to_coder(back_to_coder)
	__end__([<p>__end__</p>]):::last
	__start__ --> assemble;
	assemble --> test;
	test -.-> back_to_coder;
	test -.-> ship;
	back_to_coder --> __end__;
	ship --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



The state is a dictionary that every node reads. Each node returns only the keys it changes, and LangGraph merges
them. The conditional edge reads the state to pick the next node.

The three builds are the plain kit, which has no comets yet, and two real attempts by the Coder from a saved run of
the crew : its first attempt and its fixed one. Both attempts add the shield twist.

In [4]:
saved = json.loads(Path(data("game-studio/saved-run.json")).read_text(encoding="utf-8"))
first_reply = next(call["reply"] for call in saved["calls"] if call["node"] == "coder")
buggy_logic = max(re.findall(r"```(?:javascript|js)?\s*\n(.*?)```", first_reply, re.S), key=len)
twist = saved["board"]["design"]["twist"]

fixed_logic = saved["board"]["logic"]
builds = [("plain kit", harness.logic_of(TEMPLATE), None), ("first attempt", buggy_logic, twist), ("fixed attempt", fixed_logic, twist)]
for name, logic, build_twist in builds:
    result = graph.invoke({"logic": logic, "twist": build_twist})
    print(f"{name:14} {result['summary']:34} -> {result['verdict']}")

plain kit      10 of 19 rule tests pass           -> back to the Coder


first attempt  20 of 21 rule tests pass           -> back to the Coder


fixed attempt  21 of 21 rule tests pass           -> ship


`stream` shows the same run one node at a time, with the update each node returned. A failed test goes back to the
Coder ; in this first graph that is the end of the run.

In [5]:
for step in graph.stream({"logic": buggy_logic, "twist": twist}):
    for node, update in step.items():
        print(node, "->", {key: (f"{len(value):,} characters" if key == "html" else value) for key, value in update.items()})

assemble -> {'html': '12,790 characters'}


test -> {'passed': False, 'summary': '20 of 21 rule tests pass'}
back_to_coder -> {'verdict': 'back to the Coder'}


## b · first-multi-agent

Now the crew builds a game. Each crew member is a tool, and each tool makes one model call in a fresh conversation.
The Designer picks a twist from the menu, the Artist picks colours and draws sprites, the Coder writes the logic.
The tests are plain code : a real test run and a real bot. The model only reads the result.

In [6]:
from langchain_core.callbacks import BaseCallbackHandler


class Meter(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input = self.output = 0

    def on_llm_end(self, response, **kwargs):
        usage = response.generations[0][0].message.usage_metadata or {}
        self.calls += 1
        self.input += usage.get("input_tokens", 0)
        self.output += usage.get("output_tokens", 0)

    def __str__(self):
        return f"{self.calls} calls, {self.input:,} input tokens, {self.output:,} output tokens"


def correct(answer, expected):
    text = answer.lower()
    return all(any(alt in text for alt in item.split("|")) for item in expected)

from langchain.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

MENU = "\n".join(f"- {key} : {value['name']}. {value['pitch']} " + " ".join(value["rules"]) for key, value in TWISTS.items())
crew_llm = make_llm(max_tokens=8000)


class Sprites(BaseModel):
    star: list[str] = Field(description="7 strings of 7 characters, # filled and . empty")
    comet: list[str] = Field(description="7 strings of 7 characters, # filled and . empty")
    extra: list[str] = Field(description="7 strings of 7 characters, the twist object, # filled and . empty")


class Theme(BaseModel):
    name: str
    background: str = Field(description="dark colour, #rrggbb")
    grid: str = Field(description="colour slightly lighter than the background, #rrggbb")
    head: str = Field(description="#rrggbb")
    body: str = Field(description="#rrggbb")
    star: str = Field(description="#rrggbb")
    comet: str = Field(description="#rrggbb")
    extra: str = Field(description="#rrggbb")
    sprites: Sprites


def code_block(text):
    blocks = re.findall(r"```(?:javascript|js)?\s*\n(.*?)```", text, re.S)
    return (max(blocks, key=len) if blocks else text).replace("/* LOGIC:START */", "").replace("/* LOGIC:END */", "").strip()

The project board is a plain dictionary here. The code stays on the board, so the Producer never reads it. Each crew
member is a subagent and sees only what it receives.

In [7]:
def new_board():
    return {"logic": harness.logic_of(TEMPLATE), "theme": None, "twist": None, "report": None, "attempts": 0, "order": []}


board = new_board()


@tool
def ask_designer(brief: str) -> str:
    """Send a brief to the Designer. Returns the chosen twist and its rules."""
    board["order"].append("designer")
    reply = crew_llm.invoke([
        SystemMessage("You are the Designer at Comet Arcade. Pick the one twist from the menu that answers the brief. "
                      "Reply with the twist id on the first line, then its rules copied exactly."),
        HumanMessage(f"Brief : {brief}\n\nTwist menu :\n{MENU}")])
    board["twist"] = next((key for key in TWISTS if re.search(rf"\b{key}\b", reply.content.splitlines()[0].lower())), None)
    return reply.content


@tool
def ask_artist(idea: str) -> str:
    """Send the game's idea to the Artist. The Artist keeps its colours and sprites on the project board."""
    board["order"].append("artist")
    theme = crew_llm.with_structured_output(Theme).invoke([
        SystemMessage("You are the Artist at Comet Arcade. You pick the colours and draw sprites for a space game. "
                      "A sprite is 7 strings of exactly 7 characters : # for a filled pixel, . for an empty one. "
                      "Background and grid are dark. Head, body, star, comet and extra must stand out from the "
                      "background and from each other."),
        HumanMessage(f"Game : {idea}\nThe extra object : {board['twist']}")])
    board["theme"] = theme.model_dump()
    return f"Theme {theme.name} saved to the board."


@tool
def ask_coder(rules: str, bugs: str = "") -> str:
    """Send the twist rules, and any bugs to fix, to the Coder. The Coder keeps the code on the project board."""
    board["order"].append("coder")
    board["attempts"] += 1
    ask = (f"{RULES}\n\n## The twist\n\n{rules}\n\n## The logic block\n\n```javascript\n{board['logic']}\n```\n\n"
           "Return the complete logic block with every rule and the twist in it, in one javascript code block. "
           "Keep every helper. Keep the function names init and step.")
    if bugs:
        ask += f"\n\n## Bugs to fix\n\n{bugs}"
    reply = crew_llm.invoke([
        SystemMessage("You are the Coder at Comet Arcade. You write the pure game logic of a one-file canvas game."),
        HumanMessage(ask)])
    board["logic"] = code_block(reply.content)
    return f"Logic block saved to the board, {len(board['logic'].splitlines())} lines."


@tool
def run_game_tests() -> str:
    """Run the rule tests and the bot on the code on the project board. Returns the test report."""
    board["order"].append("tester")
    board["report"] = harness.run_tests(harness.splice(TEMPLATE, board["logic"], board["theme"]), board["twist"])
    return harness.render(board["report"])

The Producer is one agent with the crew as its tools. It decides what happens next, and it reads only what the tools
return : a short note from the Artist and the Coder, the rules from the Designer, the report from the tests.

In [8]:
from langchain.agents import create_agent

producer = create_agent(crew_llm, tools=[ask_designer, ask_artist, ask_coder, run_game_tests], system_prompt=(
    "You are the Producer at Comet Arcade, a studio of agents. You take a brief and assign the work. "
    "First ask the Designer. Then ask the Artist, with the game's idea. Then send the Designer's rules to the Coder. "
    "Then run the tests. If tests fail, send the failures back to the Coder, at most 3 Coder attempts. "
    "Do one thing at a time. When the tests pass or the attempts run out, reply in two sentences : ship or do not "
    "ship, and why."))
print(list(producer.get_graph().nodes))

['__start__', 'model', 'tools', '__end__']


We give the Producer a brief. The crew usually finishes in under a minute, and the test report decides whether the
build is good. With this kit the first build usually passes, so the fix loop often stays idle. When a test fails,
the failure goes back to the Coder.

In [9]:
import time

brief = BRIEFS[0]["brief"]
print("brief:", brief)
meter = Meter()
start = time.time()
result = producer.invoke({"messages": [HumanMessage(brief)]}, {"callbacks": [meter], "recursion_limit": 30})

for m in result["messages"]:
    for call in getattr(m, "tool_calls", []):
        print(f"{call['name']:15} <- {str(call['args'])[:100]}")
    if m.type == "tool":
        print(f"{m.name:15} -> {m.content.splitlines()[0][:100]}")
print()
print("Producer:", result["messages"][-1].content)
print("order   :", board["order"])
print("twist   :", board["twist"], " attempts:", board["attempts"])
print(meter, f" {time.time() - start:.0f} seconds")

brief: Playtesters love Comet Snake but rage-quit when one comet ends a long run. Give them one way to survive.


ask_designer    <- {'brief': 'Comet Snake is loved, but playtesters rage-quit when a single comet ends a long run. Desi
ask_designer    -> shield
ask_artist      <- {'idea': 'Comet Snake: add a clean, readable shield pickup to the existing game. The shield is a one
ask_artist      -> Theme Comet Snake — Shield Pickup saved to the board.
ask_coder       <- {'rules': "Implement the Designer's shield rules exactly: initialize shield=false. Every fourth star
ask_coder       -> Logic block saved to the board, 139 lines.
run_game_tests  <- {}
run_game_tests  -> 21 of 21 rule tests pass

Producer: Ship it: the one-use shield gives players a second chance against a comet without otherwise changing the game. All 21 rule tests pass, and the bot averages 66 stars.
order   : ['designer', 'artist', 'coder', 'tester']
twist   : shield  attempts: 1
8 calls, 5,630 input tokens, 1,865 output tokens  35 seconds


The verdict comes from the harness, whatever the model believes. We print the report, and the Artist's colours and
one of its sprites.

In [10]:
print(harness.render(board["report"]))
print()
theme = board["theme"]
print(theme["name"], {key: theme[key] for key in ("background", "head", "body", "star", "comet", "extra")})
print("\n".join(row.replace(".", "  ").replace("#", "██") for row in theme["sprites"]["comet"]))

21 of 21 rule tests pass
bot : 66.0 stars on average, best score 80

Comet Snake — Shield Pickup {'background': '#101827', 'head': '#7ee787', 'body': '#3fb950', 'star': '#ffd866', 'comet': '#ff6b6b', 'extra': '#42dcff'}
      ████    
    ████████  
██████████████
██████████████
    ████████  
      ████    
        ██    


A passing test run proves the rules. Only opening the file proves that it plays. The crew's game goes into one HTML
file under `workspace/`, and the cell below shows it in the notebook. Click the game, then steer with the arrow keys.
`R` restarts. The file also opens in any browser, on its own, with no server.

In [11]:
from html import escape

from IPython.display import display

game = harness.splice(TEMPLATE, board["logic"], board["theme"])
path = Path("workspace/comet-snake-mine.html")
path.parent.mkdir(exist_ok=True)
path.write_text(game, encoding="utf-8")
print(f"{path.as_posix()}, {len(game):,} characters")

# import webbrowser ; webbrowser.open(path.resolve().as_uri())
display({"text/html": f'<iframe srcdoc="{escape(game)}" width="770" height="620" style="border:0"></iframe>'}, raw=True)

workspace/comet-snake-mine.html, 12,841 characters


<iframe srcdoc="<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Comet Snake</title>
<style>
 html, body { margin: 0; height: 100%; background: #070b1c; color: #e6ecff; font: 16px/1.4 ui-monospace, Consolas, monospace; }
 main { display: flex; flex-direction: column; align-items: center; gap: 8px; padding: 12px; }
 canvas { border: 2px solid #28335c; border-radius: 6px; image-rendering: pixelated; }
 #hud { display: flex; gap: 24px; }
</style>
</head>
<body>
<main>
 <div id="hud"><span id="score">score 0</span><span id="pace">200 ms</span><span id="status">arrow keys steer, R restarts</span></div>
 <canvas id="game" width="720" height="540"></canvas>
</main>

<!-- The Artist's part : colours and sprites, 7 rows of 7 characters, # is a filled pixel -->
<script id="theme" type="application/json">
/* THEME:START */
{
 "name": "Comet Snake \u2014 Shield Pickup",
 "background": "#101827",
 "grid": "#1b2a3c",
 "head": "#7ee787",
 "body": "#3fb950",
 "star": "#ffd866",
 "comet": "#ff6b6b",
 "extra": "#42dcff",
 "sprites": {
 "star": [
 "...#...",
 "..###..",
 ".#####.",
 "#######",
 ".#####.",
 "..###..",
 "...#..."
 ],
 "comet": [
 "...##..",
 "..####.",
 "#######",
 "#######",
 "..####.",
 "...##..",
 "....#.."
 ],
 "extra": [
 "..###..",
 ".#...#.",
 "#..#..#",
 "#.###.#",
 "#..#..#",
 ".#...#.",
 "..###.."
 ]
 }
}
/* THEME:END */
</script>

<!-- The Coder's part : pure game logic. No DOM, no clock, no Math.random. Runs in the browser and in QuickJS. -->
<script id="logic">
/* LOGIC:START */
const COLS = 24;
const ROWS = 18;
const START_MS = 200;
const VECTORS = { up: { x: 0, y: -1 }, down: { x: 0, y: 1 }, left: { x: -1, y: 0 }, right: { x: 1, y: 0 } };
const OPPOSITE = { up: 'down', down: 'up', left: 'right', right: 'left' };

function clone(value) { return JSON.parse(JSON.stringify(value)); }
function wrap(value, size) { return (value + size) % size; }
function same(a, b) { return a.x === b.x && a.y === b.y; }

// The only source of randomness. The seed lives in the state, so a replay is identical.
function nextRandom(state) {
 state.seed = (Math.imul(state.seed, 1664525) + 1013904223) >>> 0;
 return state.seed / 4294967296;
}

function occupied(state, cell) {
 return [...state.snake, ...state.stars, ...state.comets, ...state.extras].some(item => same(item, cell));
}

// A random cell that holds nothing and is at least `distance` steps from the head.
function freeCell(state, distance = 0) {
 const head = state.snake[0];
 for (let tries = 0; tries < 500; tries++) {
 const cell = { x: Math.floor(nextRandom(state) * COLS), y: Math.floor(nextRandom(state) * ROWS) };
 if (!occupied(state, cell) && Math.abs(cell.x - head.x) + Math.abs(cell.y - head.y) >= distance) return cell;
 }
 return null;
}

function spawnStar(state) {
 const cell = freeCell(state);
 if (cell) state.stars.push(cell);
}

function spawnComet(state) {
 const cell = freeCell(state, 8);
 if (!cell) return;
 const direction = Object.values(VECTORS)[Math.floor(nextRandom(state) * 4)];
 state.comets.push({ x: cell.x, y: cell.y, dx: direction.x, dy: direction.y });
}

function spawnExtra(state, kind) {
 const cell = freeCell(state);
 if (cell) state.extras.push({ x: cell.x, y: cell.y, kind });
}

function init(seed) {
 const state = {
 snake: [{ x: 6, y: 9 }, { x: 5, y: 9 }, { x: 4, y: 9 }],
 dir: 'right', stars: [], comets: [], extras: [],
 score: 0, eaten: 0, tickMs: START_MS, ticks: 0, over: false, reason: null,
 shield: false,
 seed: (seed >>> 0) + 1,
 };
 spawnStar(state);
 spawnComet(state);
 spawnComet(state);
 return state;
}

// One tick of the game. Returns a new state and leaves `previous` untouched.
function step(previous, input) {
 if (previous.over) return previous;

 const state = clone(previous);
 if (input && VECTORS[input] && input !== OPPOSITE[state.dir]) state.dir = input;

 const vector = VECTORS[state.dir];
 const head = {
 x: wrap(state.snake[0].x + vector.x, COLS),
 y: wrap(state.snake[0].y + vector.y, ROWS),
 };

 const movedC

## Things to try

Optional.

- Add a branch to the first graph that sends a build with a low bot score to a human
- Give the Producer another brief from `BRIEFS` and play the result
- Write a brief for a twist off the menu and read what the Designer and the Coder do with it. A twist off the menu
  has no tests, so a pass says nothing about the twist
- Print `producer.get_graph().draw_mermaid()` and read the agent's graph